# JC-partial — รัน configuration เดียว

Notebook นี้รันเฉพาะ **JC-partial**, sigma_D=0.0, 30 epochs และประเมิน e20/e30 แล้วจบ
ไม่มีการเริ่ม GAN configuration ถัดไปอัตโนมัติ

1. Add Input **dataset cyclegan-paper-suite เดิม** ไม่ต้องอัปโหลด dataset ใหม่
2. เลือก GPU **P100** และเปิด Internet
3. ใช้ **Save Version → Save & Run All** เพื่อเริ่มงานเบื้องหลัง (ต่างจาก Run All ใน editor)
4. เมื่อจบ ดาวน์โหลด `paper-suite-results.zip` หรือเก็บ notebook output ไว้

**Classifier:** JC ใช้ตัวเดิมใน dataset; CP/JP เทรน classifier ของคู่นั้นก่อน GAN
ถ้าแนบ output ที่มี classifier ของคู่เดียวกันทำเสร็จแล้ว จะใช้ตัวนั้นต่อและไม่เทรนซ้ำ

**ต่อจากงานเดิม:** Add Input output ของ paper-suite ที่ต้องการเพียงหนึ่งชุด
ใช้ได้ทั้ง output คิวรวมเดิมและ output ของ notebook แยกนี้ โปรแกรมตรวจ snapshot
และต่อ checkpoint ที่มีอยู่ เฉพาะ JC-partial เท่านั้น ถ้ายังไม่มีจะเริ่มตัวนี้ใหม่
การแนบ input ต้องเป็น output ที่บันทึกไว้แล้ว ไม่ใช่ session ที่ยังรันอยู่

`selection_complete` หมายถึงตัวที่เลือกเสร็จแล้ว แม้ตัวอื่นยัง pending
`paused` หมายถึงยังไม่จบ ให้แนบ output นี้ใน session ใหม่แล้วรัน notebook ตัวเดิมต่อ
หนึ่ง configuration อาจยังใช้มากกว่าหนึ่ง session; ไม่ได้ลด epochs หรือขนาดโมเดล


In [ ]:
import time
from pathlib import Path

SESSION_STARTED = time.monotonic()
MODE = "train"  # เปลี่ยนเป็น "smoke" เพื่อทดสอบ update ของตัวนี้ก่อน
SESSION_HOURS = 10.0  # รวม setup + เผื่อ export 15 นาที
RUN_IDS = ["JC-partial"]  # รันเฉพาะตัวนี้แล้วจบ
CODE_DIR = None  # ค้นชุดข้อมูลที่แนบให้เอง
RESUME_DIR = None  # ค้น output เดิมให้เองถ้าพบชุดเดียว
ENV_PREFIX = Path("/kaggle/temp/cyclegan-tf14-env")
OUTPUT = Path("/kaggle/working/cyclegan-paper-suite")
assert MODE in ("train", "smoke")
assert 0 < SESSION_HOURS <= 10.5

In [ ]:
import json
import subprocess
import sys
import zipfile

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True).strip()
print(gpu)
if "P100" not in gpu:
    raise RuntimeError("ชุด native TF1.4 นี้ต้องใช้ P100; ไม่สลับ runtime/GPU โดยอัตโนมัติ")

def matching_manifests(base, filename):
    matches = []
    for path in Path(base).rglob(filename):
        try:
            if json.loads(path.read_text()).get("protocol") == "native-tf14-paper-selected-nine-v1":
                matches.append(path)
        except (ValueError, OSError):
            pass
    return matches

def unpack_checked(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for name in archive.namelist():
            if "\\" in name or not (target / name).resolve().is_relative_to(target.resolve()):
                raise ValueError("Unsafe ZIP member: " + name)
        archive.extractall(target)

if RESUME_DIR is None:
    resumes = matching_manifests("/kaggle/input", "suite-manifest.json")
    if not resumes:
        archives = list(Path("/kaggle/input").rglob("paper-suite-results.zip"))
        if len(archives) > 1:
            raise RuntimeError("พบหลายผลเดิม ให้ระบุ RESUME_DIR")
        if len(archives) == 1:
            unpack_checked(archives[0], Path("/kaggle/temp/suite-resume-input"))
            resumes = matching_manifests("/kaggle/temp/suite-resume-input", "suite-manifest.json")
    if len(resumes) > 1:
        raise RuntimeError("พบหลายผลเดิม ให้ระบุ RESUME_DIR ที่ต้องการ")
    if resumes:
        RESUME_DIR = resumes[0].parent

if RESUME_DIR is not None:
    RESUME_DIR = Path(RESUME_DIR)
    CODE_DIR = RESUME_DIR / "code"  # ใช้ snapshot เดิมเมื่อต่อ session
elif OUTPUT.exists():
    CODE_DIR = OUTPUT / "code"  # ต่อใน output ของ session เดิม
elif CODE_DIR is None:
    matches = matching_manifests("/kaggle/input", "bundle-manifest.json")
    if not matches:
        archives = list(Path("/kaggle/input").rglob("cyclegan-paper-suite.zip"))
        if len(archives) != 1:
            raise RuntimeError("Add Input แพ็กเกจ cyclegan-paper-suite.zip")
        unpack_checked(archives[0], Path("/kaggle/temp/suite-upload"))
        matches = matching_manifests("/kaggle/temp/suite-upload", "bundle-manifest.json")
    if len(matches) != 1:
        raise RuntimeError("พบหลายแพ็กเกจ ให้ระบุ CODE_DIR")
    CODE_DIR = matches[0].parent
CODE_DIR = Path(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, str(CODE_DIR / "kaggle"))
from suite_session import verify_bundle, verify_saved
from bootstrap import install
from suite_plan import REMAINING
if RESUME_DIR is not None:
    verify_saved(RESUME_DIR)
elif OUTPUT.exists():
    verify_saved(OUTPUT)
else:
    verify_bundle(CODE_DIR)
RUN_IDS = REMAINING if RUN_IDS is None else RUN_IDS
if not RUN_IDS or any(run not in REMAINING for run in RUN_IDS) or len(set(RUN_IDS)) != len(RUN_IDS):
    raise ValueError("RUN_IDS ต้องเป็นชื่อการทดลองที่เหลือและไม่ซ้ำ")
print("Queue:", RUN_IDS)
print("Resume:", RESUME_DIR)
legacy_python = install(ENV_PREFIX)

In [ ]:
assert RUN_IDS == ["JC-partial"], "ใช้ notebook ของ configuration ที่ต้องการ"
remaining = SESSION_HOURS * 3600 - (time.monotonic() - SESSION_STARTED)
if remaining <= 1200:
    raise RuntimeError("เวลา session เหลือไม่พอหลัง setup; ใช้ session ใหม่")
command = [sys.executable, str(CODE_DIR / "kaggle/suite_session.py"),
           "--bundle", str(CODE_DIR), "--python", str(legacy_python),
           "--output", str(OUTPUT), "--mode", MODE, "--seconds", str(remaining),
           "--runs"] + list(RUN_IDS)
if RESUME_DIR is not None:
    command += ["--resume", str(RESUME_DIR)]
subprocess.run(command, check=True)

In [ ]:
import json
from IPython.display import display, Markdown, FileLink

status = json.loads((OUTPUT / "session-status.json").read_text())
print("Configuration:", RUN_IDS[0])
print("State:", status["state"])
if status["state"] in ("selection_complete", "complete"):
    print("ตัวนี้เสร็จแล้ว — ไม่เริ่ม configuration ถัดไป")
elif status["state"] == "paused":
    print("ยังไม่จบ — เก็บ output นี้ แล้วแนบเพื่อรัน notebook ตัวเดิมต่อใน session ใหม่")
display(Markdown((OUTPUT / "PAPER-TABLES.md").read_text()))
for path in [OUTPUT / "paper-tables.csv", OUTPUT / "results.json", OUTPUT.parent / "paper-suite-results.zip"]:
    if path.exists():
        display(FileLink(str(path)))
